# Pass-rush aggregates from `player_play` (game & season)

Manual notebook for Kevin Zhang (BDB 2027 / Idea A).

**Outputs (agent-blocked):**
- `DerivedData/player_play_passrush_by_game_player.csv` — one row per `(game_id, nfl_id)` (+ season)
- `DerivedData/player_play_passrush_by_season_player.csv` — one row per `(season, nfl_id)`

Do **not** paste raw competition CSV rows into AI chats. Print column names / shape / aggregate summaries only.

## Schema source

Official competition columns are **snake_case** (see competition Data tab / `DataDescription`-style dictionary). This notebook prefers those names; camelCase BDB-2025 aliases are only fallbacks.

| Concept | Competition column |
|---|---|
| lined-up role | `lined_up_position` — keep `EDGE`, `OLB`, `INTERIOR_LINE` |
| Pass play | `pass_result` non-empty (`C`,`I`,`S`,`IN`,`R`); empty string = non-pass. Fallback: non-null `blitzing` / `sack` (populated on dropbacks) |
| `time_to_pressure` | `time_to_pressure` |
| `time_to_qb_hurry` | `time_to_qb_hurry` |
| `sack` | `sack` (0 / 0.5 / 1) |
| `player_get_off` | `player_get_off` |
| pressure event | non-null `time_to_pressure` (rusher credited) |
| `blitzing` | `blitzing` |
| `unblocked_pressure` | `unblocked_pressure` |
| `quick_pressure` | `quick_pressure` |
| season / REG | `games.season`, `games.season_type` |
| player name / roster pos | `players.display_name`, `players.nfl_position` |

Join keys: `game_id`, `nfl_id`. Null marker in files is often the string `NA`.


In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Paths — edit if your local layout differs
# ---------------------------------------------------------------------------
REPO_ROOT = Path("../..").resolve()
DATA_DIR = REPO_ROOT / "Data"            # competition CSVs (agent-ignored)
DERIVED_DIR = REPO_ROOT / "DerivedData"  # outputs (agent-ignored)
# Optional local copy of the competition data dictionary (markdown)
DATA_DESCRIPTION = DATA_DIR / "DataDescription.md"
if not DATA_DESCRIPTION.is_file():
    DATA_DESCRIPTION = REPO_ROOT / "DataDescription.md"

PLAYER_PLAY = DATA_DIR / "player_play.csv"
GAMES = DATA_DIR / "games.csv"
PLAYERS = DATA_DIR / "players.csv"

OUT_GAME = DERIVED_DIR / "player_play_passrush_by_game_player.csv"
OUT_SEASON = DERIVED_DIR / "player_play_passrush_by_season_player.csv"

LINED_UP_KEEP = {"EDGE", "OLB", "INTERIOR_LINE"}
PASS_RESULTS = {"C", "I", "S", "IN", "R"}  # non-empty pass_result values
STORE_TIME_LISTS = True  # set False to skip JSON time-list columns

print("REPO_ROOT:", REPO_ROOT)
print("DATA_DIR exists:", DATA_DIR.is_dir())
print("DataDescription:", "OK" if DATA_DESCRIPTION.is_file() else "not found (using built-in snake_case map)")
for p in (PLAYER_PLAY, GAMES, PLAYERS):
    print(f"  {p.name}: {'OK' if p.is_file() else 'MISSING'} -> {p}")


In [ ]:
def _norm(name: str) -> str:
    return re.sub(r"[^a-z0-9]", "", str(name).lower())


def resolve_column(columns, *candidates, required: bool = False) -> str | None:
    """Match a column by exact name, casefold, or alphanumeric-normalized form."""
    cols = list(columns)
    by_cf = {c.casefold(): c for c in cols}
    by_norm = {_norm(c): c for c in cols}
    for cand in candidates:
        if cand in cols:
            return cand
        if cand.casefold() in by_cf:
            return by_cf[cand.casefold()]
        if _norm(cand) in by_norm:
            return by_norm[_norm(cand)]
    if required:
        raise KeyError(f"Required column not found. Tried: {candidates}. Have: {cols}")
    return None


def headers_only(path: Path) -> list[str]:
    return list(pd.read_csv(path, nrows=0).columns)


# Header-only verification (no row dump)
for label, path in [("player_play", PLAYER_PLAY), ("games", GAMES), ("players", PLAYERS)]:
    if not path.is_file():
        print(f"[skip] {label}: file missing")
        continue
    cols = headers_only(path)
    print(f"{label}: {len(cols)} columns")
    print(cols)


In [ ]:
# Resolve schema from headers (still no row contents printed)
assert PLAYER_PLAY.is_file(), f"Missing {PLAYER_PLAY} — set DATA_DIR at top of notebook"
pp_cols = headers_only(PLAYER_PLAY)
games_cols = headers_only(GAMES) if GAMES.is_file() else []
player_cols = headers_only(PLAYERS) if PLAYERS.is_file() else []

# BDB 2027 official names are snake_case; camelCase listed only as legacy aliases
COL = {
    "game_id": resolve_column(pp_cols, "game_id", "gameId", required=True),
    "play_id": resolve_column(pp_cols, "play_id", "playId", required=True),
    "nfl_id": resolve_column(pp_cols, "nfl_id", "nflId", required=True),
    "team_abbr": resolve_column(pp_cols, "team_abbr", "teamAbbr", "club"),
    "lined_up": resolve_column(
        pp_cols, "lined_up_position", "linedUpPosition", "linedUpPos", required=True
    ),
    "pass_result": resolve_column(pp_cols, "pass_result", "passResult"),
    "had_dropback": resolve_column(pp_cols, "had_dropback", "hadDropback", "is_dropback", "isDropback"),
    "time_to_pressure": resolve_column(
        pp_cols, "time_to_pressure", "timeToPressureAsPassRusher", "timeToPressure"
    ),
    "time_to_qb_hurry": resolve_column(
        pp_cols,
        "time_to_qb_hurry",
        "timeToHurryAsPassRusher",
        "timeToQbHurryAsPassRusher",
        "timeToQuarterbackHurry",
        "timeToHurry",
    ),
    "player_get_off": resolve_column(
        pp_cols, "player_get_off", "getOffTimeAsPassRusher", "get_off_time", "getOffTime"
    ),
    "caused_pressure": resolve_column(pp_cols, "caused_pressure", "causedPressure", "pressure"),
    "sack": resolve_column(pp_cols, "sack", "sackYardsAsDefense", "got_sack"),
    "blitzing": resolve_column(pp_cols, "blitzing", "is_blitzing", "isBlitzing", "didBlitz"),
    "unblocked_pressure": resolve_column(
        pp_cols, "unblocked_pressure", "unblockedPressure", "causedUnblockedPressure"
    ),
    "quick_pressure": resolve_column(
        pp_cols, "quick_pressure", "quickPressure", "causedQuickPressure"
    ),
}

GAMES_GAME_ID = resolve_column(games_cols, "game_id", "gameId", required=bool(games_cols)) if games_cols else None
GAMES_SEASON = resolve_column(games_cols, "season") if games_cols else None
GAMES_WEEK = resolve_column(games_cols, "week") if games_cols else None
GAMES_TYPE = (
    resolve_column(games_cols, "season_type", "seasonType", "game_type", "gameType")
    if games_cols
    else None
)
PLAYERS_NFL_ID = resolve_column(player_cols, "nfl_id", "nflId", required=bool(player_cols)) if player_cols else None
PLAYERS_NAME = (
    resolve_column(player_cols, "display_name", "displayName", "player_name", "playerName", "name")
    if player_cols
    else None
)
PLAYERS_POS = (
    resolve_column(player_cols, "nfl_position", "position", "nflPosition") if player_cols else None
)

print("Resolved player_play mapping:")
for k, v in COL.items():
    print(f"  {k:22s} -> {v}")
print("games:", GAMES_GAME_ID, GAMES_SEASON, GAMES_WEEK, GAMES_TYPE)
print("players:", PLAYERS_NFL_ID, PLAYERS_NAME, PLAYERS_POS)

missing_core = [k for k in ("time_to_pressure", "player_get_off", "sack") if COL[k] is None]
if missing_core:
    print("WARNING: core pass-rush columns missing:", missing_core)
if COL["pass_result"] is None and COL["had_dropback"] is None:
    print("WARNING: no pass_result/had_dropback — will use dropback-field presence proxy.")


In [ ]:
# Load only needed columns (still: do not display raw row contents)
usecols = [c for c in COL.values() if c is not None]
usecols = list(dict.fromkeys(usecols))

# Preserve literal string values; competition uses "NA" as a null marker in many columns
pp = pd.read_csv(
    PLAYER_PLAY,
    usecols=usecols,
    dtype={COL["lined_up"]: "string"},
    keep_default_na=True,
    na_values=["NA", "NaN", "nan", ""],
    low_memory=False,
)
# Re-read lined_up with NA preserved as a category label if present as string NA
# (pandas may have turned "NA" into <NA>; that is OK — those rows are not EDGE/OLB/INTERIOR_LINE)
print("player_play loaded:", pp.shape, "| columns:", list(pp.columns))

games = None
if GAMES.is_file() and GAMES_SEASON is not None and GAMES_GAME_ID is not None:
    g_usecols = [GAMES_GAME_ID, GAMES_SEASON]
    if GAMES_WEEK:
        g_usecols.append(GAMES_WEEK)
    if GAMES_TYPE:
        g_usecols.append(GAMES_TYPE)
    games = pd.read_csv(GAMES, usecols=g_usecols, na_values=["NA"], low_memory=False)
    print("games loaded:", games.shape)
else:
    print("WARNING: games.csv missing/unusable — season aggregates need games.season.")

players = None
if PLAYERS.is_file() and PLAYERS_NFL_ID is not None:
    p_usecols = [PLAYERS_NFL_ID]
    if PLAYERS_NAME:
        p_usecols.append(PLAYERS_NAME)
    if PLAYERS_POS:
        p_usecols.append(PLAYERS_POS)
    players = pd.read_csv(PLAYERS, usecols=p_usecols, na_values=["NA"], low_memory=False)
    print("players loaded:", players.shape)


In [ ]:
def to_numeric_series(s: pd.Series) -> pd.Series:
    if s.dtype == bool:
        return s.astype(float)
    if pd.api.types.is_numeric_dtype(s):
        return s.astype(float)
    mapped = s.replace({True: 1, False: 0, "True": 1, "False": 0, "true": 1, "false": 0})
    return pd.to_numeric(mapped, errors="coerce")


def binary_event(s: pd.Series) -> pd.Series:
    """Nonzero numeric/bool -> 1.0; NaN/0 -> 0.0 for rate denominators."""
    x = to_numeric_series(s)
    return (x.fillna(0) != 0).astype(float)


# Filters -------------------------------------------------------------------
lined = pp[COL["lined_up"]].astype("string").str.strip().str.upper()
lined = lined.str.replace("-", "_", regex=False).str.replace(" ", "_", regex=False)
mask_pos = lined.isin(LINED_UP_KEEP)

if COL["pass_result"] is not None:
    pr = pp[COL["pass_result"]].astype("string").str.strip().str.upper()
    mask_pass = pr.isin(PASS_RESULTS)
    pass_rule = f"{COL['pass_result']} in {sorted(PASS_RESULTS)}"
elif COL["had_dropback"] is not None:
    drop = to_numeric_series(pp[COL["had_dropback"]])
    mask_pass = drop.fillna(0) != 0
    pass_rule = f"{COL['had_dropback']} != 0"
else:
    # Dropback proxy: defender rush/pressure fields are null on non-dropbacks
    proxy = [COL[k] for k in ("blitzing", "sack", "unblocked_pressure", "time_to_pressure") if COL[k]]
    mask_pass = False
    for c in proxy:
        mask_pass = mask_pass | pp[c].notna()
    pass_rule = "proxy: non-null blitzing/sack/pressure fields"

filt = pp.loc[mask_pos & mask_pass].copy()
print("filter lined_up in", sorted(LINED_UP_KEEP), "& pass rule:", pass_rule)
print("rows in -> kept:", len(pp), "->", len(filt), "(counts only; no row dump)")

# Join season / names
if games is not None:
    filt = filt.merge(
        games,
        how="left",
        left_on=COL["game_id"],
        right_on=GAMES_GAME_ID,
        suffixes=("", "_game"),
    )
    if GAMES_TYPE and GAMES_TYPE in filt.columns:
        st = filt[GAMES_TYPE].astype("string").str.upper()
        if st.isin(["REG", "REGULAR", "REGULAR_SEASON"]).any():
            before = len(filt)
            filt = filt.loc[st.isin(["REG", "REGULAR", "REGULAR_SEASON"])].copy()
            print(f"REG filter via {GAMES_TYPE}: {before} -> {len(filt)}")

if players is not None:
    filt = filt.merge(
        players,
        how="left",
        left_on=COL["nfl_id"],
        right_on=PLAYERS_NFL_ID,
        suffixes=("", "_player"),
    )


In [ ]:
# Feature engineering on filtered play rows
work = filt.copy()
idx = work.index


def col_or_nan(colname: str | None) -> pd.Series:
    if colname is None:
        return pd.Series(np.nan, index=idx, dtype="float64")
    return to_numeric_series(work[colname])


work["time_to_pressure"] = col_or_nan(COL["time_to_pressure"])
work["time_to_qb_hurry"] = col_or_nan(COL["time_to_qb_hurry"])
work["player_get_off"] = col_or_nan(COL["player_get_off"])

# Pressure event: prefer explicit caused_pressure; else credit when time_to_pressure is present
if COL["caused_pressure"]:
    work["pressure_event"] = binary_event(work[COL["caused_pressure"]])
else:
    work["pressure_event"] = work["time_to_pressure"].notna().astype(float)

# Hurry event: non-null hurry time
work["hurry_event"] = work["time_to_qb_hurry"].notna().astype(float)

# Sack: column is already 0 / 0.5 / 1 on dropbacks (null elsewhere)
work["sack_event"] = binary_event(work[COL["sack"]]) if COL["sack"] else pd.Series(0.0, index=idx)
# Keep fractional sack credit for counts
work["sack_credit"] = col_or_nan(COL["sack"]).fillna(0.0) if COL["sack"] else pd.Series(0.0, index=idx)

work["blitzing"] = binary_event(work[COL["blitzing"]]) if COL["blitzing"] else pd.Series(np.nan, index=idx)
work["unblocked_pressure"] = (
    binary_event(work[COL["unblocked_pressure"]]) if COL["unblocked_pressure"] else pd.Series(np.nan, index=idx)
)
work["quick_pressure"] = (
    binary_event(work[COL["quick_pressure"]]) if COL["quick_pressure"] else pd.Series(np.nan, index=idx)
)

work["lined_up_position"] = lined.loc[idx].astype("string")

print("engineered columns ready; null rates / means only (no row dump):")
for c in [
    "time_to_pressure",
    "time_to_qb_hurry",
    "player_get_off",
    "pressure_event",
    "hurry_event",
    "sack_event",
    "blitzing",
    "unblocked_pressure",
    "quick_pressure",
]:
    s = work[c]
    print(
        f"  {c:22s} non-null={int(s.notna().sum())}  "
        f"mean={float(s.mean(skipna=True)) if s.notna().any() else float('nan')}"
    )


In [ ]:
def json_nonnull_list(series: pd.Series) -> str:
    vals = [float(x) for x in series.dropna().tolist()]
    return json.dumps(vals, separators=(",", ":"))


def aggregate(df: pd.DataFrame, group_keys: list[str]) -> pd.DataFrame:
    rows = []
    grouped = df.groupby(group_keys, dropna=False, sort=False)
    for key, g in grouped:
        if not isinstance(key, tuple):
            key = (key,)
        rec = dict(zip(group_keys, key))
        rec["n_plays"] = int(len(g))

        for time_col, mean_name, n_name, list_name in [
            ("time_to_pressure", "mean_time_to_pressure", "n_time_to_pressure", "times_to_pressure_json"),
            ("time_to_qb_hurry", "mean_time_to_qb_hurry", "n_time_to_qb_hurry", "times_to_qb_hurry_json"),
            ("player_get_off", "mean_player_get_off", "n_player_get_off", "player_get_off_times_json"),
        ]:
            s = g[time_col]
            rec[mean_name] = float(s.mean(skipna=True)) if s.notna().any() else np.nan
            rec[n_name] = int(s.notna().sum())
            if STORE_TIME_LISTS:
                rec[list_name] = json_nonnull_list(s)

        for ev_col, rate_name, count_name in [
            ("pressure_event", "pressure_rate", "n_pressures"),
            ("hurry_event", "hurry_rate", "n_hurries"),
            ("sack_event", "sack_rate", "n_sack_plays"),
        ]:
            s = g[ev_col].fillna(0)
            rec[rate_name] = float(s.mean())
            rec[count_name] = int(s.sum())

        # Fractional sack credits (0.5 shared sacks)
        rec["n_sacks"] = float(g["sack_credit"].sum()) if "sack_credit" in g.columns else float(rec["n_sack_plays"])

        for opt_col, rate_name, count_name in [
            ("blitzing", "blitz_rate", "n_blitzes"),
            ("unblocked_pressure", "unblocked_pressure_rate", "n_unblocked_pressures"),
            ("quick_pressure", "quick_pressure_rate", "n_quick_pressures"),
        ]:
            s = g[opt_col]
            if s.notna().any():
                filled = s.fillna(0)
                rec[rate_name] = float(filled.mean())
                rec[count_name] = int(filled.sum())
            else:
                rec[rate_name] = np.nan
                rec[count_name] = 0

        if "lined_up_position" in g.columns and g["lined_up_position"].notna().any():
            rec["primary_lined_up_position"] = g["lined_up_position"].mode(dropna=True).iloc[0]
        else:
            rec["primary_lined_up_position"] = pd.NA

        if COL["team_abbr"] and COL["team_abbr"] in g.columns:
            rec["team_abbr"] = (
                g[COL["team_abbr"]].mode(dropna=True).iloc[0]
                if g[COL["team_abbr"]].notna().any()
                else pd.NA
            )
        if PLAYERS_NAME and PLAYERS_NAME in g.columns:
            rec["display_name"] = (
                g[PLAYERS_NAME].mode(dropna=True).iloc[0] if g[PLAYERS_NAME].notna().any() else pd.NA
            )
        if PLAYERS_POS and PLAYERS_POS in g.columns:
            rec["nfl_position"] = (
                g[PLAYERS_POS].mode(dropna=True).iloc[0] if g[PLAYERS_POS].notna().any() else pd.NA
            )

        rows.append(rec)
    return pd.DataFrame(rows)


game_keys = [COL["game_id"], COL["nfl_id"]]
if GAMES_SEASON and GAMES_SEASON in work.columns:
    game_keys = [GAMES_SEASON, COL["game_id"], COL["nfl_id"]]

by_game = aggregate(work, game_keys)
print("by_game shape:", by_game.shape)

if GAMES_SEASON and GAMES_SEASON in work.columns:
    by_season = aggregate(work, [GAMES_SEASON, COL["nfl_id"]])
else:
    raise RuntimeError("Cannot build season aggregates without games.season — join games.csv first.")
print("by_season shape:", by_season.shape)

print("by_game columns:", list(by_game.columns))
print("by_season columns:", list(by_season.columns))
print(by_game[["n_plays", "pressure_rate", "mean_time_to_pressure", "mean_player_get_off"]].describe())


In [ ]:
DERIVED_DIR.mkdir(parents=True, exist_ok=True)
by_game.to_csv(OUT_GAME, index=False)
by_season.to_csv(OUT_SEASON, index=False)
print("Wrote:", OUT_GAME)
print("Wrote:", OUT_SEASON)
print("Done. Keep DerivedData/ out of git and out of agent-readable paths.")


## Output column summary

**Keys:** `season`, `game_id` (game table), `nfl_id`; optional `display_name`, `nfl_position`, `team_abbr`, `primary_lined_up_position`.

**Volume:** `n_plays`.

**Times (mean over non-NaN + n + optional JSON lists):**
`mean_time_to_pressure`, `n_time_to_pressure`, `times_to_pressure_json`,
`mean_time_to_qb_hurry`, `n_time_to_qb_hurry`, `times_to_qb_hurry_json`,
`mean_player_get_off`, `n_player_get_off`, `player_get_off_times_json`.

**Events:** `pressure_rate` / `n_pressures`, `hurry_rate` / `n_hurries`,
`sack_rate` / `n_sack_plays` / `n_sacks` (fractional credits),
optional `blitz_rate`, `unblocked_pressure_rate`, `quick_pressure_rate` (+ counts).
